# Where this actually fits

> A working classifier in eight lines, then the only paragraph of vocabulary you need, then the honest answer to why none of this worked until recently.

Read this chapter at `/learn/01-where-this-fits/`. Exported from `src/content/chapters/01-where-this-fits.mdx` — edit there, not here.


Here is a machine learning model. Press **Run**.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

digits = load_digits()                       # 1797 hand-written digits, 8x8 pixels
X_train, X_valid, y_train, y_valid = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=0)

model = LogisticRegression(max_iter=5000).fit(X_train, y_train)
print(f"accuracy on digits it has never seen: {model.score(X_valid, y_valid):.1%}")

That is not a toy in the dishonest sense. It is a real classifier, trained on real
data, scored on data it was not trained on, and it is right about roughly
nineteen out of twenty hand-written digits. It took eight lines and about two
seconds.

Take a moment on what is *missing* from those eight lines. Nowhere did anyone
write down what a 7 looks like. Nobody enumerated the strokes, or handled the
case where the crossbar is missing, or wrote a rule about the little hook some
people put on a 9. There is no `if` statement anywhere in this program about
digits at all.

You did not write the rules. You supplied examples, and a procedure searched for
the rules. That inversion is the entire subject.

## The inversion

The programs you have written until now have this shape:

<div class="table-scroll">

| | You provide | The computer produces |
|---|---|---|
| **Ordinary program** | rules + input | output |
| **Machine learning** | input + output | **rules** |

</div>

That is it. That is the whole conceptual leap, and everything else in this
tutorial is mechanism.

You already know why the second row is worth having. You have written the first
row for years, and you know exactly which problems it is bad at — the ones where
you cannot state the rule. You can specify a JSON parser completely. You cannot
specify "is this photo a cat", "is this review angry", "is this transaction
fraud", not because you do not know the answer when you see one, but because the
knowledge is not in a form you can type.

It is closest to the difference between writing a function and writing a
*constraint*. You know `fn is_seven(pixels: &[u8]) -> bool` is the signature you
want. You have no idea what goes in the body. Machine learning is a way of
filling in that body by search, given a large number of `(input, expected)`
pairs and a way to score how badly a candidate body is doing.

## The vocabulary, once

Four words get used interchangeably in public and mean quite different things.
Here they are, from largest to smallest, and then we will mostly stop talking
about them.

**Artificial intelligence** is the field, and it is old — the name was coined at
a workshop in 1956. It includes chess engines built entirely from hand-written
search, expert systems built from hand-written rules, and everything below. It
is an *aspiration*, not a technique. Nobody sensible describes their work as "AI"
when talking to another practitioner.

**Machine learning** is the subset of AI where the rules are fitted to data
instead of written. Logistic regression is machine learning. So is a decision
tree, so is the spam filter that shipped in 2002. Most of it involves no neural
networks whatsoever.

**Deep learning** is the subset of machine learning that uses neural networks
with many layers. "Deep" literally means "has a lot of layers stacked up". It is
the part that got extremely good after about 2012, and it is where nearly all
recent excitement lives.

**A model** is the fitted thing itself — the function with its numbers filled in.
`model` in the code above is a model. A file of weights is a model. When someone
says "we deployed the model", they mean they shipped a function.

The regrettable consequence of this nesting is that a linear regression from 1805
is, technically and correctly, artificial intelligence. When a press release says
"AI-powered", it is compatible with a spreadsheet formula. Being able to ask
"which layer of that nest do you mean" is a genuinely useful professional skill.

Three more words you will need in the next paragraph:

- **Training** — the process of finding the numbers. Expensive, done once, offline.
- **Inference** — running the finished model on new input. Cheap, done constantly.
- **Parameters** (or *weights*) — the numbers that got found. `LogisticRegression`
  above found 650 of them. GPT-scale models have hundreds of billions.

## What the eight lines actually did

Let us take the model apart, because everything later in this tutorial is a
variation on what is inside it.

In [ ]:
w = model.coef_          # the learned weights
b = model.intercept_     # the learned offsets
print("weights:", w.shape, " offsets:", b.shape)
print("total learned numbers:", w.size + b.size)

Ten rows of 64 weights, plus ten offsets. One row per digit, one weight per pixel.
That is the model, in its entirety — 650 floating-point numbers. Everything
`load_digits` knows that has been captured is in there.

To classify an image, the model computes a score for each of the ten digits by
multiplying every pixel by that digit's weight and adding them up, then picks the
highest. Let us do it by hand, without scikit-learn, to prove there is nothing
else hiding.

In [ ]:
import numpy as np

image = X_valid[0]                    # one 8x8 digit, flattened to 64 numbers
scores = image @ w.T + b              # ten scores, one per digit
print("scores :", np.round(scores, 1))
print("predicted:", scores.argmax(), " actual:", y_valid[0])

One matrix multiply, one addition, one
argmax. That is *inference*. The entire deployed cost of a
model like this is a handful of multiplications — which is why the expensive part
was never running it, it was finding those 650 numbers in the first place.

Each row of `w` is a template — a 64-number picture of what that digit tends to
look like. The  between an image and a template is large
when the bright pixels of the image line up with the large weights of the
template. So "which digit is this" becomes "which template does this image agree
with most", and agreement is measured by a dot product.

You can literally look at the templates as images:

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(7, 3))
for digit, ax in enumerate(axes.flat):
    ax.imshow(w[digit].reshape(8, 8), cmap="RdBu_r")
    ax.set_title(str(digit), fontsize=9); ax.axis("off")
plt.tight_layout()

Red is "pixels here argue for this digit", blue is "pixels here argue against
it". The 0 template is a ring: bright in the middle counts against it. The 1
template is a vertical stripe. Nobody told it that. It fell out of 1257 examples
and some arithmetic.

This is also, precisely, the limitation. A single template per class cannot
represent "a 7 with a crossbar *or* a 7 without one" — it has to average them,
and the average of two different-looking 7s is a blurrier 7 that matches neither
well. Fixing that is what layers are for, and that is
[Chapter 8](/learn/08-neural-networks/).

## Why didn't this work until recently?

This is the question worth sitting with, because the answer is not the one most
people assume, and it tells you where the field is going.

**The ideas are old.** The perceptron — a single artificial neuron, trainable by
gradient descent — was built as physical hardware in 1958. Backpropagation, the
algorithm that makes deep networks trainable and which we will build by hand in
[Chapter 9](/learn/09-backpropagation/), was popularised in 1986. Convolutional
networks for reading digits were working in production, sorting cheques for the
US Postal Service, in the early 1990s. Long short-term memory, which handled
sequences for two decades, is from 1997.

So the honest answer to "why not until now" is not *we thought of it*. Three
other things had to arrive.

**Data.** A model fits itself to examples, so it needs examples. The dataset that
broke the field open was ImageNet: 14 million labelled photographs, assembled
between 2007 and 2009, largely by paying people on Mechanical Turk to label
images one at a time. Before the internet made collection cheap and crowdsourcing
made labelling cheap, nobody had a million labelled anything.

**Compute — and specifically the *right shape* of compute.** You saw above that
inference is a matrix multiply. Training is a great many more of them. Graphics
cards happen to be machines built to do enormous numbers of parallel
multiply-accumulates, for reasons entirely about rendering triangles. When people
started running neural networks on them around 2009, the same experiment got
roughly fifty times faster overnight. An idea that takes six months to test is
not an idea you can iterate on; an idea that takes three days is.

**A handful of unglamorous tricks.** Better weight initialisation. ReLU instead
of sigmoid, so gradients survive many layers ([why](/appendix/math/#relu)).
Dropout. Batch normalisation. Adam. Individually each is a paragraph of
arithmetic. Collectively they moved deep networks from "theoretically trainable"
to "trainable on a Tuesday".

The moment those three met is dateable: **September 2012**, when AlexNet won the
ImageNet competition with an error rate of 15.3% against the runner-up's 26.2%.
That is not an incremental win. In a mature benchmark, a ten-point gap is the
sound of a field changing direction.

The pattern generalises, and it is the most useful thing on this page for
predicting what happens next. Almost every "breakthrough" in this field is an old
idea meeting newly-sufficient scale. Transformers (2017) made attention — an idea
from 2014 — cheap enough to scale. Diffusion models are 2015 mathematics that
became practical in 2021. When you read that something is impossible, it is worth
asking whether it is impossible or merely currently expensive.

## When the answer is not machine learning

Because the counter-examples are where judgement lives, and nobody teaches them.

**When you can write the rule.** Tax calculation is defined in legislation. Do
not learn it from examples; you will get 99.4% accuracy on a problem where 100%
was available for free, and you will be unable to explain the 0.6%.

**When being wrong is unacceptable and unexplainable.** These models are
statistical. They are wrong sometimes and cannot always tell you why. If a wrong
answer means a wrong medication, the model is at best an input to a human
decision.

**When you have no data.** Two hundred examples of a rare event is not a training
set, it is an anecdote. Sometimes the correct project is spending six months
building the pipeline that collects data, and shipping the model next year.

**When a lookup table would do.** An enormous amount of shipped "AI" is a
`GROUP BY` with better marketing. Try the boring thing first and measure it. It
is your baseline, and if the model cannot beat it, you have learned something
cheaply.

The expensive failure mode in industry is not choosing the wrong architecture. It
is spending four months on a model for a problem where the data could never have
supported an answer. The skill that prevents it is framing, and it is
[Chapter 3](/learn/03-the-shape-of-problems/).

## The map

Everything above is one branch of a larger picture. Here is the whole thing.
You will not understand most of it yet — that is expected, and it is the point.
Come back after every part and watch it fill in.

Three questions hang off machine learning, and conflating them is the main source
of early confusion. *What feedback do you get?* (supervised, unsupervised,
self-supervised, reinforcement.) *What shape is the learned function?* (linear
model, tree, neural network.) *What makes the fitting work?* (losses, gradients,
validation.) Every project answers all three, and they are independent choices.

## Where you are going

By the end of the second week you will have built, by hand and from nothing but
NumPy arrays: a linear model, a gradient descent
optimiser, a neural network, and the backpropagation pass that trains it. Then
you will do all of it again in PyTorch in a tenth of the code, and understand
exactly what the library is doing on your behalf — which is the only durable way
to debug it.

You will also be able to read the first page of a paper and know which of these
boxes it lives in. That, more than any particular architecture, is what makes the
rest of the field learnable on your own.

Tomorrow: Python, at the speed of someone who already knows how to program.